## TSS Pandas Challenge #2 - Data Cleaning, Merging & Analysis

Reference: https://www.kaggle.com/competitions/tss-pandas-challenge-2/overview

Welcome to TSS Pandas Challenge #2, organized by The Software Society (TSS).

In Part 1, you worked with a dataset that was already clean and ready to analyze. In the real world, however, data rarely looks that perfect.

This challenge picks up where Part 1 left off. You will work with messy data, clean it, combine two separate datasets, and perform analysis using Pandas — following the same workflow a data analyst uses in real-world projects.

This challenge focuses on intermediate Pandas concepts such as:
- Handling missing values
- Removing duplicate records
- Standardizing inconsistent text
- Merging datasets
- Working with dates
- Grouping and analyzing data

## 🎯 Objective
Your goal is to clean and combine the provided datasets and then use Pandas to answer six analytical questions correctly.

You are expected to write and run your own Python code rather than manually calculating the answers.

## Datasets

### 📁 File 1: student-data-v2.csv
This file contains information about students, similar to Part 1.

However, this dataset contains some intentional real-world data quality issues:
- Missing values
- At least one duplicate row
- Inconsistent formatting in department names
- Extra spaces in some department values
- Different capitalization styles

### 📁 File 2: mentor-feedback.csv
This file contains mentor feedback scores for students.

Not every student in the student dataset has a matching feedback record. Some students have not been reviewed yet.

This is intentional and will allow you to practice handling missing data after performing a left join.


### Follow the data-cleaning workflow in order:

**Load → Inspect → Clean → Merge → Analyze**

A recommended workflow is:

```text
1. Load the datasets
        ↓
2. Inspect the data
        ↓
3. Check missing values
        ↓
4. Remove duplicate rows
        ↓
5. Standardize department names
        ↓
6. Merge student and mentor datasets
        ↓
7. Analyze feedback scores
        ↓
8. Convert and analyze dates
        ↓
9. Prepare the submission CSV
```

### 🟢 Q1 — Data Cleaning: Missing Values — 15 Points
- Load student-data-v2.csv using Pandas.
- Determine the total number of missing values across the entire dataset.
- You must add the missing values from every column, not just one particular column.

In [ ]:
# Importar librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cargar data de alumnos
df_alumnos = pd.read_csv("./challenge/student-data-v2.csv")
df_alumnos.head()

In [ ]:
# Mostrar el conteo total de valores nulos de todas las columnas por orden descendente
df_alumnos.isnull().sum().sort_values(ascending=False)

### Respuesta 1: Columnas con valores nulos
- attendance => 3
- python_marks  => 2

### 🟢 Q2 — Data Cleaning: Duplicates — 15 Points

- The dataset contains at least one exact duplicate row.
- Remove all exact duplicate rows and then count how many rows remain.

In [ ]:
# Mostrar filas duplicadas del dataset
df_alumnos[df_alumnos.duplicated()]

In [ ]:
# Remover duplicados
df_alumnos = df_alumnos.drop_duplicates()
# Verificar que no queden duplicados
df_alumnos[df_alumnos.duplicated()]

### Repuesta 2
- Filas duplicadas => 1 (student_id = S004)

### 🟡 Q3 — Data Cleaning: Standardizing Text — 20 Points
The department column contains inconsistent formatting.

Using the deduplicated dataset from Q2:
- Remove leading and trailing whitespace from the department column.
- Convert all department values to uppercase.
- Count how many students belong to the "AIML" department after standardization.

In [ ]:
# Normalizar datos de la columna department
df_alumnos['department'] = df_alumnos['department'].str.upper().str.strip()
df_alumnos.head()

In [ ]:
# Contamos a los alumnos del departamento de AIML
df_alumnos[df_alumnos['department'] == 'AIML'].shape[0]

In [ ]:
# Realziamos el mismo ejercicios anterior pero usando groupby, solo contando el departo AIML
df_alumnos.groupby('department').size()['AIML']

### Respuesta 3
- Cantidad de alumnos del departamento 'AIML' => 10

### 🟡 Q4 — Merging Datasets — 15 Points
Using your cleaned dataset from Q3, merge it with mentor-feedback.csv using the student_id column.
You must use a left join.

- A left join ensures that every student from the main student dataset is kept, even if they do not have a matching mentor feedback record.
- After merging, count how many students have a missing (NaN) feedback_score.

In [ ]:
# Cargando el dataset 'mentor-feedback.csv'
df_mentor_feedback = pd.read_csv('./challenge/mentor-feedback.csv')
df_mentor_feedback.head()

In [ ]:
# Realizar el merge (left join)
df_merged = pd.merge(df_alumnos, df_mentor_feedback, how='left', on='student_id')
df_merged.head()

In [ ]:
# Contamos los valores nulos del mentor
df_merged['mentor'].isna().sum()

### Respuesta 4
- El número de estudiantes sin mentos es => 5

### 🔴 Q5 — GroupBy on Merged Data — 20 Points
Using the merged dataset from Q4, calculate the average feedback_score for each department.
- Students with missing feedback scores should be automatically ignored when calculating the average.
- Then identify the highest department average feedback score.
- Round your answer to the nearest whole number**.

In [ ]:
# Eliminando filas que no tienen feedback
df_scored = df_merged[df_merged['mentor'].notna()]
df_scored.shape

In [ ]:
# Agrupando por departamento y ponderando el feedback de los mentores
df_scored.groupby('department')['feedback_score'].mean().sort_values(ascending=False)

In [ ]:
# Redondeando hacia arriba los valores promedio del feedback de los mentores
# usamos np.ceil para redondear hacia arriba
np.ceil(df_scored.groupby('department')['feedback_score'].mean()).astype(int).sort_values(ascending=False)

### Respuesta 5
- El departamento con score más alto es => ECE (con 8 puntos)

### 🔴 Q6 — Dates — 15 Points
- The join_date column is currently stored as text in the following format: DD-MM-YYYY
- Convert the join_date column into a proper Pandas datetime format.
- Then count how many students joined during August 2023.
- The date range should be: 1 August 2023 → 31 August 2023

In [ ]:
# Crear un columna adicional llamada periodo que represente el año y mes de la columna join_date que tiene formato DD-MM-YYYY
df_merged['periodo'] = pd.to_datetime(df_merged['join_date'], format='%d-%m-%Y').dt.to_period('M')
df_merged.head()

In [ ]:
# Agrupamos por periodo y contamos el nuemero de registros del periodo 2023-08
df_merged.groupby('periodo').size()['2023-08']

### Respuesta 6
- La cantidad de alumnos afiliados en el mes de agosto es => 10